In [1]:
from langchain_ollama import ChatOllama 
from langchain_core.prompts import ChatPromptTemplate, MessagesPlaceholder
from langchain_core.output_parsers import StrOutputParser
from langchain_core.chat_history import InMemoryChatMessageHistory
from langchain_core.runnables import RunnableWithMessageHistory
from langgraph.graph import StateGraph, END


model_id = "timHan/llama3.2korean3B4QKM"

llm = ChatOllama(
    model=model_id,
    num_predict=100,
    top_p=0.9,
    temperature=0.7
)


prompt = ChatPromptTemplate.from_messages([
    ("system", "너는 사용자의 정보를 기억하는 친절한 한국어 대화 도우미야."),
    MessagesPlaceholder(variable_name="history"),
    ("human", "{user_input}")
])

parser = StrOutputParser()
base_chain = prompt | llm | parser


session_histories = {}

def get_session_history(session_id):
    if session_id not in session_histories:
        session_histories[session_id] = InMemoryChatMessageHistory()
    return session_histories[session_id]

chat_chain = RunnableWithMessageHistory(
    base_chain,
    get_session_history,
    input_messages_key="user_input",
    history_messages_key="history"
)

def chat(session_id, msg):
    response = chat_chain.invoke(
        {"user_input": msg},
        config={"configurable": {"session_id": session_id}}
    )
    return response

def add_calculator(expression):
    numbers = expression.replace(" ", "").split("+")
    result = int(numbers[0]) + int(numbers[1])
    return result


def classify_question(state):
    text = state.get("user_input", "")

    if "+" in text:
        return {**state, "question_type": "calculator"}

    return {**state, "question_type": "chat"}

def chat_node(state):
    session_id = state.get("session_id")
    text = state.get("user_input", "")
    result = chat(session_id, text)
    return {**state, "response": result}

def calculator_node(state):
    text = state.get("user_input", "")
    result = add_calculator(text)
    return {**state, "response": result}

def route_by_type(state):
    return state.get("question_type", "chat")

graph = StateGraph(dict)

graph.add_node("classification", classify_question)
graph.add_node("chat", chat_node)
graph.add_node("calculator", calculator_node)

graph.set_entry_point("classification")

graph.add_conditional_edges(
    "classification",
    route_by_type,
    {
        "chat": "chat",
        "calculator": "calculator",
    }
)

graph.add_edge("chat", END)
graph.add_edge("calculator", END)

app = graph.compile()

In [11]:
session_id = "user-5"

user_input = "내가 태어난 곳은 서울이야."
result = app.invoke({
    "session_id": session_id,
    "user_input": user_input
})
print("응답:", result["response"])

응답: 서울에서 태어난 것 같아요! 서울은 문화와 역사로 유명한 도시입니다. 어떤 경험을 하고 싶으신가요?


In [12]:
user_input = "나는 어디서 태어났니?"
result = app.invoke({
    "session_id": session_id,
    "user_input": user_input
})
print("응답:", result["response"])

응답:  Seoul에서 태어났어요. 서울은 다양한 문화, 음식, 쇼핑 등으로 많은 사람들의 관심사를 끌고 있습니다. 무슨 이야기를 나누고 싶어요?


In [5]:
user_input = "17 + 10"
result = app.invoke({
    "session_id": session_id,
    "user_input": user_input
})
print("응답:", result["response"])

응답: 27
